# Hiding a Payload in Model Weights: LSB vs. Value-Mapping

**Team 8 — Part A (embedding).** Two ways to hide the **same** harmless payload in a small network's weights,
on one shared baseline model:

- **Method A — LSB substitution:** overwrite the low byte of successive weights (EvilModel / StegoNet).
- **Method B — Value-mapping:** leave the weights untouched and store an *index map* pointing to weights that
  already match the payload (StegoNet, Liu et al., ACSAC 2020, §4.2.2).

Each method has its own results cell; §4 compares them side by side. The model class and MNIST pipeline reuse
the Module 6 *Information Hiding* example by Angelica Liguori. Payloads are harmless (text and random bytes).

In [1]:
import os, struct, hashlib, copy, random
import numpy as np
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

seed = 101
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Target model (shared baseline)

Set `TARGET` below. `"fc"` builds/reloads a tiny fully-connected net; `"pretrained"` loads the Module 6 CNN
from `net.pt`. Both methods use the same model and baseline.

In [2]:
# Two model classes: our small fully-connected net, and the Module 6 CNN (only for the net.pt fallback).
class NetFC(nn.Module):
    # 784 -> 32 -> 10
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(self.flatten(x))))


class NetCNN(nn.Module):
    # Module 6 example network; matches the shipped net.pt state_dict.
    def __init__(self, n_channels=1, h=28, w=28, last=64, hidden=8, n_classes=10):
        super().__init__()
        self.embedder = nn.Sequential(
            nn.Conv2d(n_channels, last // 2, 3, stride=2), nn.ReLU(),
            nn.Conv2d(last // 2, last, 3, stride=2), nn.ReLU(),
            nn.Conv2d(last, last, 3, stride=2), nn.ReLU(),
        )
        size = self.embedder(torch.rand(1, n_channels, h, w)).shape[2]
        self.fc1 = nn.Linear(last * size * size, last)
        self.fc2 = nn.Linear(last, hidden)
        self.out = nn.Linear(hidden, n_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.embedder(x).flatten(1)
        x = self.fc2(self.relu(self.fc1(x)))
        return self.out(self.relu(x))


transform = transforms.Compose([transforms.ToTensor()])
train_set = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_set = datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256)


def accuracy(net):
    net.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (net(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total


def quick_train(net, epochs=3):
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    net.train()
    for e in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss_fn(net(x), y).backward()
            opt.step()
        print(f"  epoch {e + 1}/{epochs} done")

In [3]:
TARGET = "fc"   # "fc" = our small net; "pretrained" = load net.pt (Module 6 CNN)
FC_PATH = "pretrained_fc.pt"   # trained FC weights, saved once and reloaded for the demo

if TARGET == "fc":
    model = NetFC().to(device)
    if os.path.exists(FC_PATH):
        model.load_state_dict(torch.load(FC_PATH, weights_only=True, map_location=device))
        print(f"Loaded trained FC model from {FC_PATH}")
    else:
        print("Training the small fully-connected net...")
        quick_train(model, epochs=3)
        torch.save(model.state_dict(), FC_PATH)
        print(f"Saved trained FC model to {FC_PATH}")
elif TARGET == "pretrained":
    model = NetCNN().to(device)
    model.load_state_dict(torch.load("net.pt", weights_only=True, map_location=device))
else:
    raise ValueError("TARGET must be 'fc' or 'pretrained'")

clean_sd = copy.deepcopy(model.state_dict())   # keep an untouched copy
capacity = sum(p.numel() for p in clean_sd.values())
print(f"Model: {TARGET} | parameters (= byte capacity): {capacity:,}")
baseline = accuracy(model)
print(f"Baseline accuracy: {baseline:.2f}%")

Loaded trained FC model from pretrained_fc.pt
Model: fc | parameters (= byte capacity): 25,450


Baseline accuracy: 94.60%


## 2. Method A — LSB substitution

Pack each weight big-endian and **overwrite its lowest byte** with one payload byte (`n_bytes` deep for more
capacity). The low byte holds the least-significant mantissa bits, so the value changes by at most ~2⁻¹⁵ and
accuracy is essentially unaffected. End of payload is marked with an escape sequence (`deadbeef`); no side
information is needed to extract. The results cell hides a text payload and 1 KB of random bytes.

In [4]:
ESCAPE = b"\xde\xad\xbe\xef"


def embed_payload(sd, payload, n_bytes=1, escape=ESCAPE):
    # Writes payload+escape into the LOWEST n_bytes of successive weights (state_dict order).
    # n_bytes=1 is stealthy (no accuracy loss); larger n_bytes raises capacity but hurts accuracy.
    # Returns n_slots = number of weights used (the first n_slots are the embedded ones --
    # the hook a later 'freeze & retrain' step would use).
    data = payload + escape
    capacity = sum(p.numel() for p in sd.values()) * n_bytes
    if len(data) > capacity:
        raise ValueError(f"payload {len(data)} B exceeds capacity {capacity} B (n_bytes={n_bytes})")
    k = 0
    n_slots = 0
    with torch.no_grad():
        for p in sd.values():
            flat = p.view(-1)
            for i in range(flat.numel()):
                if k >= len(data):
                    return n_slots
                chunk = data[k:k + n_bytes]
                k += len(chunk)
                n_slots += 1
                packed = bytearray(struct.pack("!f", flat[i].item()))
                packed[4 - len(chunk):] = chunk          # overwrite the lowest len(chunk) bytes
                flat[i] = struct.unpack("!f", bytes(packed))[0]
    return n_slots


def extract_payload(sd, n_bytes=1, escape=ESCAPE):
    out = bytearray()
    for p in sd.values():
        for v in p.view(-1):
            out.extend(struct.pack("!f", v.item())[4 - n_bytes:])
            if out[-len(escape):] == escape:
                return bytes(out[:-len(escape)])
    return bytes(out)   # escape never found

In [5]:
# --- Text payload: embed, re-check accuracy, read it back ---
text = b"Team 8 FoC demo: this stands in for a malware payload. Harmless text only."

sd_text = copy.deepcopy(clean_sd)
n = embed_payload(sd_text, text)
model.load_state_dict(sd_text)
acc_after = accuracy(model)
recovered = extract_payload(sd_text)

print(f"Text payload: {len(text)} B into {n} weights ({100 * n / capacity:.2f}% of capacity)")
print(f"Accuracy after embedding: {acc_after:.2f}%")
print(f"Recovered text: {recovered.decode()}")
print(f"Text round-trip OK: {recovered == text}")

model.load_state_dict(clean_sd)   # restore the clean model

# --- Random payload: embed, read back, verify by hash ---
blob = random.randbytes(1024)
sd_blob = copy.deepcopy(clean_sd)
embed_payload(sd_blob, blob)
recovered_blob = extract_payload(sd_blob)

print()
print(f"Random payload: {len(blob)} B")
print(f"sha256 original : {hashlib.sha256(blob).hexdigest()}")
print(f"sha256 recovered: {hashlib.sha256(recovered_blob).hexdigest()}")
assert recovered_blob == blob, "random-bytes round-trip failed"
print("Random round-trip OK: True")

Text payload: 74 B into 78 weights (0.31% of capacity)
Accuracy after embedding: 94.60%
Recovered text: Team 8 FoC demo: this stands in for a malware payload. Harmless text only.
Text round-trip OK: True

Random payload: 1024 B
sha256 original : 4971a9aaf10bc5cbd4c3ac0eccf3579629381a848e8c668cdf47ad53bdf8ada8
sha256 recovered: 4971a9aaf10bc5cbd4c3ac0eccf3579629381a848e8c668cdf47ad53bdf8ada8
Random round-trip OK: True


### Deeper embedding: capacity vs. fidelity

Method A can store more by overwriting more than one byte per weight (`n_bytes`). At `n_bytes=1`, even filling
**every** weight leaves accuracy untouched — the point of LSB hiding. Writing deeper reaches more significant
mantissa/exponent bits, so capacity grows but accuracy eventually drops, while the payload still extracts back
perfectly.

In [6]:
for n_bytes in (1, 2, 3):
    cap_nb = capacity * n_bytes
    big = random.randbytes(cap_nb - len(ESCAPE))          # fill the whole capacity at this depth
    sd_big = copy.deepcopy(clean_sd)
    embed_payload(sd_big, big, n_bytes=n_bytes)
    model.load_state_dict(sd_big)
    recovered_big = extract_payload(sd_big, n_bytes=n_bytes)
    print(f"n_bytes={n_bytes}: payload {len(big):6,} B (~{cap_nb / 1024:.1f} KB) | "
          f"accuracy {accuracy(model):.2f}% | round-trip OK {recovered_big == big}")

model.load_state_dict(clean_sd)   # restore the clean model

n_bytes=1: payload 25,446 B (~24.9 KB) | accuracy 94.60% | round-trip OK True


n_bytes=2: payload 50,896 B (~49.7 KB) | accuracy 94.59% | round-trip OK True


n_bytes=3: payload 76,346 B (~74.6 KB) | accuracy 88.04% | round-trip OK True


<All keys matched successfully>

## 3. Method B — Value-mapping

For each payload byte, **search** for a weight whose lowest mantissa byte already equals it (the same byte
Method A would overwrite) and record that weight's index. Extraction reads those bytes back via the **index
map**. When a match exists — almost always, since a model has far more weights than the 256 byte values — the
weight is left untouched, so the model stays byte-for-byte identical. The price is the index map (one index
per payload byte), which is larger than the payload itself. The results cell hides the same two payloads.

In [7]:
def low_bytes(sd):
    # Lowest mantissa byte of every weight, in state_dict traversal order (uint8).
    parts = []
    for p in sd.values():
        a = p.detach().cpu().numpy().ravel().astype(np.float32)
        parts.append(a.view(np.uint8).reshape(-1, 4)[:, 0])   # [:, 0] = least-significant byte (little-endian)
    return np.concatenate(parts)


def _set_low_byte(sd, g, b):
    # Overwrite the low byte of the global-index-g weight (used only by the nearest fallback).
    off = 0
    for p in sd.values():
        n = p.numel()
        if g < off + n:
            flat = p.view(-1)
            packed = bytearray(struct.pack("!f", flat[g - off].item()))
            packed[-1] = b
            flat[g - off] = struct.unpack("!f", bytes(packed))[0]
            return
        off += n


def value_map_embed(sd, payload):
    # Map each payload byte to a weight whose low byte equals it. Returns (index_map, n_modified).
    vals = low_bytes(sd)
    buckets = {b: list(np.flatnonzero(vals == b)) for b in range(256)}
    cursor = {b: 0 for b in range(256)}
    index_map = np.empty(len(payload), dtype=np.int64)
    n_modified = 0
    for k, b in enumerate(payload):
        hits = buckets[b]
        if hits:                                             # exact match -> no change to the model
            index_map[k] = hits[cursor[b] % len(hits)]
            cursor[b] += 1
        else:                                                # nearest fallback -> overwrite one weight
            nearest = min((v for v in range(256) if buckets[v]), key=lambda v: abs(v - b))
            g = int(buckets[nearest][0])
            _set_low_byte(sd, g, b)
            index_map[k] = g
            n_modified += 1
    return index_map, n_modified


def value_map_extract(sd, index_map):
    # Read the low byte of each mapped weight, in order.
    return bytes(low_bytes(sd)[index_map].astype(np.uint8).tolist())


idx_bytes = (capacity.bit_length() + 7) // 8        # bytes needed to store one weight index
print(f"One weight index needs {idx_bytes} bytes (capacity {capacity:,} weights)")

One weight index needs 2 bytes (capacity 25,450 weights)


In [8]:
# --- Text payload: value-map it, check the model, read it back ---
text = b"Team 8 FoC demo: this stands in for a malware payload. Harmless text only."
sd_text = copy.deepcopy(clean_sd)
imap, modified = value_map_embed(sd_text, text)
model.load_state_dict(sd_text)
recovered = value_map_extract(sd_text, imap)

print(f"Text payload: {len(text)} B | weights modified: {modified} | index map: {len(imap) * idx_bytes} B")
print(f"Accuracy after mapping: {accuracy(model):.2f}%  (baseline {baseline:.2f}%)")
print(f"Recovered text: {recovered.decode()}")
print(f"Text round-trip OK: {recovered == text}")

model.load_state_dict(clean_sd)

# --- Random payload: value-map, read back, verify by hash ---
blob = random.randbytes(1024)
sd_blob = copy.deepcopy(clean_sd)
imap_b, modified_b = value_map_embed(sd_blob, blob)
recovered_blob = value_map_extract(sd_blob, imap_b)

print()
print(f"Random payload: {len(blob)} B | weights modified: {modified_b} | index map: {len(imap_b) * idx_bytes} B")
print(f"sha256 original : {hashlib.sha256(blob).hexdigest()}")
print(f"sha256 recovered: {hashlib.sha256(recovered_blob).hexdigest()}")
assert recovered_blob == blob, "random-bytes round-trip failed"
print("Random round-trip OK: True")

Text payload: 74 B | weights modified: 0 | index map: 148 B


Accuracy after mapping: 94.60%  (baseline 94.60%)
Recovered text: Team 8 FoC demo: this stands in for a malware payload. Harmless text only.
Text round-trip OK: True

Random payload: 1024 B | weights modified: 0 | index map: 2048 B
sha256 original : 780a351e5067ec6b30dc03aca36dc33092c7e99d952df047234bc947e1ed02cd
sha256 recovered: 780a351e5067ec6b30dc03aca36dc33092c7e99d952df047234bc947e1ed02cd
Random round-trip OK: True


## 4. Side by side

The same payload, hidden both ways. LSB changes the low byte of every weight it uses but needs no side
information; value-mapping changes no weights (so accuracy is exactly the baseline) but must ship an index
map larger than the payload.

In [9]:
payload = b"Team 8 FoC demo: this stands in for a malware payload. Harmless text only."

# Method A: LSB substitution
sd_lsb = copy.deepcopy(clean_sd)
n_lsb = embed_payload(sd_lsb, payload, n_bytes=1)
model.load_state_dict(sd_lsb)
acc_lsb = accuracy(model)
ok_lsb = extract_payload(sd_lsb, n_bytes=1) == payload

# Method B: value-mapping
sd_vm = copy.deepcopy(clean_sd)
imap, mod_vm = value_map_embed(sd_vm, payload)
model.load_state_dict(sd_vm)
acc_vm = accuracy(model)
ok_vm = value_map_extract(sd_vm, imap) == payload

model.load_state_dict(clean_sd)

print(f"Same {len(payload)} B payload | baseline accuracy {baseline:.2f}%\n")
print(f"{'method':<16}{'weights changed':>16}{'side info':>12}{'accuracy':>11}{'round-trip':>12}")
print(f"{'LSB':<16}{n_lsb:>16}{'none':>12}{acc_lsb:>10.2f}%{str(ok_lsb):>12}")
print(f"{'value-mapping':<16}{mod_vm:>16}{str(len(imap) * idx_bytes) + ' B':>12}{acc_vm:>10.2f}%{str(ok_vm):>12}")

Same 74 B payload | baseline accuracy 94.60%

method           weights changed   side info   accuracy  round-trip
LSB                           78        none     94.60%        True
value-mapping                  0       148 B     94.60%        True


## How to run

1. `cd CyberSec-class-project && source .venv/bin/activate` (dependencies pinned in `requirements.txt`).
2. Open this notebook and **Run All** top to bottom.
3. Default `TARGET = "fc"` reuses `pretrained_fc.pt` (trains it once, ~30 s, if missing). Set
   `TARGET = "pretrained"` to use the Module 6 `net.pt` instead.

Expected: both methods round-trip the same payloads. LSB changes the low byte of each used weight (accuracy
essentially unchanged); value-mapping changes **0** weights (accuracy equals the baseline) but needs an index
map larger than the payload. §4 shows the contrast. `net.pt` is never modified.